In [1]:
import os
import shutil
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, SubsetRandomSampler

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    confusion_matrix, accuracy_score, classification_report,
    f1_score, precision_score, recall_score
)

import kagglehub

## Config

Each entry in `MODEL_CONFIGS` is a pretrained torchvision backbone. `input_size` is the spatial resolution that architecture expects (299 for Inception v3, 224 for everything else here).

In [2]:
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 1e-4
NUM_CLASSES = 7

# name -> (constructor, expected input size)
MODEL_CONFIGS = [
    {"name": "inception_v3",        "fn": models.inception_v3,        "input_size": 299},
    {"name": "densenet121",         "fn": models.densenet121,         "input_size": 224},
    {"name": "shufflenet_v2_x2_0",  "fn": models.shufflenet_v2_x2_0,  "input_size": 224},
    {"name": "resnet18",            "fn": models.resnet18,            "input_size": 224},
    {"name": "regnet_y_400mf",      "fn": models.regnet_y_400mf,      "input_size": 224},
    {"name": "mobilenet_v3_small",  "fn": models.mobilenet_v3_small,  "input_size": 224},
    {"name": "googlenet",           "fn": models.googlenet,           "input_size": 224},
    {"name": "efficientnet_v2_s",   "fn": models.efficientnet_v2_s,   "input_size": 224},
    {"name": "alexnet",             "fn": models.alexnet,             "input_size": 224},
]

## Download FER2013

In [3]:
path = kagglehub.dataset_download("msambare/fer2013")
print("Path to dataset files:", path)

target_dir = "/content/fer2013"
os.makedirs(target_dir, exist_ok=True)

for item_name in os.listdir(path):
    source_path = os.path.join(path, item_name)
    destination_path = os.path.join(target_dir, item_name)
    if os.path.isdir(source_path):
        shutil.copytree(source_path, destination_path, dirs_exist_ok=True)
    else:
        shutil.copy(source_path, destination_path)

Using Colab cache for faster access to the 'fer2013' dataset.
Path to dataset files: /kaggle/input/fer2013


## Transforms

Images are read as grayscale then **duplicated to 3 channels** (`Grayscale(num_output_channels=3)`) so every pretrained backbone can consume them unmodified. `build_transforms(size)` resizes to whatever resolution a given architecture expects.

In [4]:
def build_transforms(input_size, train=True):
    common = [
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((input_size, input_size)),
    ]
    if train:
        aug = [
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(20),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ]
    else:
        aug = []

    tail = [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ]
    return transforms.Compose(common + aug + tail)

## Train/val split + class weights

This only needs the folder structure/labels, not pixel data, so it's independent of which backbone's input size we'll use later.

In [5]:
data_root = os.path.join(os.getcwd(), "fer2013")

dataset_for_split = datasets.ImageFolder(os.path.join(data_root, "train"))
targets = np.array(dataset_for_split.targets)
class_names = dataset_for_split.classes
print("Dataset classes:", class_names)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(np.zeros(len(targets)), targets))

train_targets = targets[train_idx]
class_counts = Counter(train_targets)
total_samples = sum(class_counts.values())

class_weights = torch.zeros(NUM_CLASSES, dtype=torch.float32)
for i in range(NUM_CLASSES):
    if class_counts[i] > 0:
        class_weights[i] = total_samples / (NUM_CLASSES * class_counts[i])

for i, name in enumerate(class_names):
    print(f"{name} weight:\t{class_weights[i]:.4f}")

Dataset classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
angry weight:	1.0266
disgust weight:	9.4011
fear weight:	1.0012
happy weight:	0.5684
neutral weight:	0.8260
sad weight:	0.8491
surprise weight:	1.2933


## Loader factory

Builds fresh train/val loaders at whatever `input_size` a given backbone needs (reusing the same stratified `train_idx`/`val_idx` split for every model, only the resolution differs).

In [6]:
def make_loaders(input_size, batch_size=BATCH_SIZE):
    train_tf = build_transforms(input_size, train=True)
    val_tf = build_transforms(input_size, train=False)

    train_dataset = datasets.ImageFolder(os.path.join(data_root, "train"), transform=train_tf)
    val_dataset = datasets.ImageFolder(os.path.join(data_root, "train"), transform=val_tf)

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                               sampler=SubsetRandomSampler(train_idx))
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                             sampler=SubsetRandomSampler(val_idx))
    return train_loader, val_loader


def make_test_loader(input_size, batch_size=BATCH_SIZE):
    test_tf = build_transforms(input_size, train=False)
    test_dataset = datasets.ImageFolder(os.path.join(data_root, "test"), transform=test_tf)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
    return test_loader, test_dataset.classes

## Backbone wrapper

Wraps any torchvision classification model: loads ImageNet weights, replaces its final classification layer with one sized for `num_classes`, and reuses the same train/test loop shape as before. Handles the few architectures (Inception v3, GoogLeNet) that return an auxiliary-logits namedtuple during training by just taking `.logits`.

In [7]:
class BackboneClassifier(nn.Module):
    def __init__(self, backbone_fn, num_classes=NUM_CLASSES, lr=1e-4, epochs=30):
        super().__init__()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.backbone = backbone_fn(weights="DEFAULT")
        self._replace_head(num_classes)

        self.optimizer = optim.Adam(self.parameters(), lr=lr, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=epochs)

        self.to(self.device)

    def _replace_head(self, num_classes):
        # Different torchvision architectures name/structure their final
        # classification layer differently, so we handle the two common
        # shapes: a plain `fc` Linear, or a `classifier` (Linear or
        # Sequential ending in Linear).
        if hasattr(self.backbone, "fc") and isinstance(self.backbone.fc, nn.Linear):
            in_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Linear(in_features, num_classes)
            return

        if hasattr(self.backbone, "classifier"):
            classifier = self.backbone.classifier
            if isinstance(classifier, nn.Linear):
                in_features = classifier.in_features
                self.backbone.classifier = nn.Linear(in_features, num_classes)
                return
            if isinstance(classifier, nn.Sequential):
                for i in reversed(range(len(classifier))):
                    if isinstance(classifier[i], nn.Linear):
                        in_features = classifier[i].in_features
                        classifier[i] = nn.Linear(in_features, num_classes)
                        return

        raise ValueError(f"Don't know how to replace the head of {type(self.backbone).__name__}")

    def forward(self, x):
        x = x.to(self.device)
        out = self.backbone(x)
        # Inception v3 / GoogLeNet return a namedtuple with an aux head
        # during training; we only care about the main logits.
        if hasattr(out, "logits"):
            out = out.logits
        return out

    def _loss_(self, outputs, targets, class_weights=None):
        targets = targets.to(self.device)
        if class_weights is not None:
            class_weights = class_weights.to(self.device)
            loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        else:
            loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
        return loss_fn(outputs, targets)

    def train_(self, train_loader, epochs=30, class_weights=None, val_loader=None, ckpt_path="best_model.pth"):
        history = {"train_loss": [], "val_loss": [], "val_acc": []}
        best_val_acc = 0
        for epoch in range(epochs):
            self.train()
            total_train_loss = 0

            for x, targets in train_loader:
                x, targets = x.to(self.device), targets.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.forward(x)
                loss = self._loss_(outputs, targets, class_weights)

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.parameters(), 5.0)
                self.optimizer.step()

                total_train_loss += loss.item()

            avg_train_loss = total_train_loss / len(train_loader)
            history["train_loss"].append(avg_train_loss)

            if val_loader is not None:
                self.eval()
                total_val_loss = 0
                correct = 0
                total = 0

                with torch.no_grad():
                    for x_val, targets_val in val_loader:
                        x_val, targets_val = x_val.to(self.device), targets_val.to(self.device)
                        outputs_val = self.forward(x_val)

                        val_loss = self._loss_(outputs_val, targets_val, class_weights)
                        total_val_loss += val_loss.item()

                        _, predicted = torch.max(outputs_val, 1)
                        total += targets_val.size(0)
                        correct += (predicted == targets_val).sum().item()

                avg_val_loss = total_val_loss / len(val_loader)
                val_acc = correct / total
                history["val_loss"].append(avg_val_loss)
                history["val_acc"].append(val_acc)

                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    torch.save(self.state_dict(), ckpt_path)

                print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | "
                      f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")
                self.scheduler.step()
            else:
                print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f}")

        if val_loader is not None:
            self.load_state_dict(torch.load(ckpt_path))

        return history

    def test_(self, data_loader):
        self.eval()
        correct = 0
        total = 0
        all_preds, all_targets, all_logits = [], [], []

        with torch.no_grad():
            for x, targets in data_loader:
                x, targets = x.to(self.device), targets.to(self.device)
                outputs = self.forward(x)

                _, predicted = torch.max(outputs, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())
                all_logits.extend(outputs.cpu().numpy())

        return correct / total, np.array(all_preds), np.array(all_targets), np.array(all_logits)

## Train every backbone

Each model gets its own loaders (built at its required `input_size`) and its own checkpoint filename, so runs no longer overwrite each other's `best_model.pth`.

In [8]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

trained_models = {}
histories = {}

for cfg in MODEL_CONFIGS:
    name, fn, input_size = cfg["name"], cfg["fn"], cfg["input_size"]
    print("=" * 60)
    print(f"Training {name} (input size {input_size})")
    print("=" * 60)

    train_loader, val_loader = make_loaders(input_size)

    net = BackboneClassifier(fn, num_classes=NUM_CLASSES, lr=LEARNING_RATE, epochs=EPOCHS)
    history = net.train_(
        train_loader,
        epochs=EPOCHS,
        class_weights=class_weights,
        val_loader=val_loader,
        ckpt_path=f"best_model_{name}.pth",
    )

    trained_models[name] = net
    histories[name] = history
    torch.save(net.state_dict(), f"model_weights_{name}.pth")

Training inception_v3 (input size 299)
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 191MB/s] 


Epoch [1/30] Train Loss: 1.7965 | Val Loss: 1.5840 | Val Acc: 0.5557


KeyboardInterrupt: 

## Training curves

Overlays every backbone's per-epoch train loss, val loss, and val accuracy so you can compare convergence speed and spot overfitting.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for name, hist in histories.items():
    epochs_ran = range(1, len(hist["train_loss"]) + 1)
    axes[0].plot(epochs_ran, hist["train_loss"], label=name)
    if hist["val_loss"]:
        axes[1].plot(epochs_ran, hist["val_loss"], label=name)
        axes[2].plot(epochs_ran, hist["val_acc"], label=name)

axes[0].set_title("Train Loss")
axes[1].set_title("Val Loss")
axes[2].set_title("Val Accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Evaluation

Runs `.test_()` for every trained backbone and reports the same metrics/plots as before, one block per model.

In [ ]:
def evaluate_and_plot(name, net, cfg):
    input_size = cfg["input_size"]
    test_loader, test_class_names = make_test_loader(input_size)

    accuracy, all_predicted, all_targets, all_outputs = net.test_(test_loader)

    f1 = f1_score(all_targets, all_predicted, average="weighted")
    precision = precision_score(all_targets, all_predicted, average="weighted")
    recall = recall_score(all_targets, all_predicted, average="weighted")

    print("=" * 40)
    print(f"  Model         : {name}")
    print(f"  Test Accuracy : {accuracy:.4f}")
    print(f"  F1 Score      : {f1:.4f}")
    print(f"  Precision     : {precision:.4f}")
    print(f"  Recall        : {recall:.4f}")
    print("=" * 40)

    cm = confusion_matrix(all_targets, all_predicted)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=test_class_names, yticklabels=test_class_names)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"Confusion Matrix (Raw Counts) — {name}")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=test_class_names, yticklabels=test_class_names,
                vmin=0, vmax=1)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"Confusion Matrix (Row Normalised) — {name}")
    plt.tight_layout()
    plt.show()

    report_dict = classification_report(all_targets, all_predicted,
                                         target_names=test_class_names, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df = report_df.drop(columns=["support"])
    report_df = report_df.loc[test_class_names]

    plt.figure(figsize=(10, 6))
    sns.heatmap(report_df, annot=True, cmap="viridis", fmt=".2f", vmin=0, vmax=1)
    plt.title(f"Classification Report Heatmap — {name}")
    plt.ylabel("Class")
    plt.xlabel("Metrics")
    plt.tight_layout()
    plt.show()

    return {"name": name, "accuracy": accuracy, "f1": f1, "precision": precision, "recall": recall}


results = []
for cfg in MODEL_CONFIGS:
    name = cfg["name"]
    results.append(evaluate_and_plot(name, trained_models[name], cfg))

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
results_df